In [ ]:
!pip install transformers torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
train_path = "/content/drive/MyDrive/archive/DATASET/TRAIN"
test_path  = "/content/drive/MyDrive/archive/DATASET/TEST"

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

In [ ]:
train_dataset = datasets.ImageFolder(train_path, transform=transform_train)
test_dataset  = datasets.ImageFolder(test_path, transform=transform_test)

class_names = train_dataset.classes

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
import torch
from transformers import ViTForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(class_names),
    ignore_mismatched_sizes=True
)

model.to(device)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([5, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([5])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name or "classifier" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
import torch.nn.functional as F

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.7)

model.train()

for epoch in range(6):
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(pixel_values=images)
        loss = F.cross_entropy(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    torch.save(model.state_dict(), "/content/drive/MyDrive/yoga_best.pth")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1, Loss: 98.2721
Epoch 2, Loss: 65.0483
Epoch 3, Loss: 43.8810
Epoch 4, Loss: 33.6776
Epoch 5, Loss: 27.7052
Epoch 6, Loss: 23.9827


In [ ]:
model.eval()

correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(pixel_values=images)
        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Accuracy:", (correct/total)*100)

Accuracy: 95.95744680851064


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/yoga_checkpoint.pth")

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/yoga_best.pth")

In [ ]:
import torch

torch.save(model.state_dict(), "/content/drive/MyDrive/yoga_best.pth")

In [ ]:
import torch

torch.save(model.state_dict(), "/content/drive/MyDrive/yoga_best.pth")
print("Saved correctly!")

Saved correctly!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'KAVYA THATAVARTHI (9).pdf', 'idcard.pdf', 'kavyaphoto.jpg', 'Screenshot_20260224_222905_PhonePe.jpg', 'archive', 'yoga_best.pth', 'yoga_checkpoint.pth']


In [ ]:
import torch
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=3,
    ignore_mismatched_sizes=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([3, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([3])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=5,   # 🔥 CHANGE THIS
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([5, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/yoga_best.pth"))

<All keys matched successfully>

In [ ]:
import torch

torch.save(model.state_dict(), "/content/drive/MyDrive/yoga_best.pth")
print("Final model saved ✅")

Final model saved ✅


In [ ]:
# Run this in your Colab notebook
from torchvision import datasets
import os

# Path to your training data
train_path = "/content/drive/MyDrive/archive/DATASET/TRAIN"

# Load the dataset to get class names
train_dataset = datasets.ImageFolder(train_path)

print("=" * 60)
print("YOUR MODEL'S ACTUAL CLASSES:")
print("=" * 60)

for i, class_name in enumerate(train_dataset.classes):
    print(f"{i}: {class_name}")

print("=" * 60)
print(f"Total classes: {len(train_dataset.classes)}")

# Save to file
with open("actual_classes.txt", "w") as f:
    for class_name in train_dataset.classes:
        f.write(f"{class_name}\n")

print("\n✅ Saved to actual_classes.txt")

# Download the file
from google.colab import files
files.download("actual_classes.txt")

YOUR MODEL'S ACTUAL CLASSES:
0: downdog
1: goddess
2: plank
3: tree
4: warrior2
Total classes: 5

✅ Saved to actual_classes.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>